In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
import sklearn 
import xgboost 
from pathlib import Path 

In [2]:
path=Path('/home/yazid/agro/1_Traditional_ml/data_fetching')
my_list=[pd.read_csv(df) for df in path.glob('*.csv')]

In [3]:
df=pd.concat(my_list,ignore_index=True)

In [4]:
len(df)


58440

In [5]:
df.head()

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,surface_pressure_mean,cloud_cover_mean,dew_point_2m_mean,wind_speed_10m_max,wind_gusts_10m_max,wind_direction_10m_dominant,shortwave_radiation_sum,et0_fao_evapotranspiration,city,Unnamed: 0
0,2010-01-01,14.3,17.5,10.7,0.0,0.0,0.0,76,1015.0,40,9.8,13.4,24.8,224,8.61,1.64,Casablanca,NaN
1,2010-01-02,13.4,18.5,9.2,0.0,0.0,0.0,81,1013.7,24,9.9,7.6,19.4,114,12.45,1.87,Casablanca,NaN
2,2010-01-03,15.7,20.7,12.3,0.0,0.0,0.0,77,1013.4,93,11.4,13.6,27.0,195,10.62,2.03,Casablanca,NaN
3,2010-01-04,16.1,19.1,14.5,2.3,2.3,11.0,76,1007.1,41,11.9,29.7,64.4,224,7.84,2.07,Casablanca,NaN
4,2010-01-05,14.6,15.4,13.3,4.0,4.0,14.0,84,1007.5,63,11.9,27.4,64.1,240,6.35,1.43,Casablanca,NaN


In [6]:
df.isna().sum()

date                               0
temperature_2m_mean                0
temperature_2m_max                 0
temperature_2m_min                 0
precipitation_sum                  0
rain_sum                           0
precipitation_hours                0
relative_humidity_2m_mean          0
surface_pressure_mean              0
cloud_cover_mean                   0
dew_point_2m_mean                  0
wind_speed_10m_max                 0
wind_gusts_10m_max                 0
wind_direction_10m_dominant        0
shortwave_radiation_sum            0
et0_fao_evapotranspiration         0
city                               0
Unnamed: 0                     46752
dtype: int64

In [7]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [8]:
df['date']=pd.to_datetime(df['date'])
df['date'].dtype

dtype('<M8[us]')

In [9]:
df['month']=df['date'].dt.month

Let's do some cyclical encoding on our month values:

In [10]:
df['month_cos']=np.cos(2*np.pi*df['month']/12)
df['month_sin']=np.sin(2*np.pi*df['month']/12)

In [11]:
list(df.columns)

['date',
 'temperature_2m_mean',
 'temperature_2m_max',
 'temperature_2m_min',
 'precipitation_sum',
 'rain_sum',
 'precipitation_hours',
 'relative_humidity_2m_mean',
 'surface_pressure_mean',
 'cloud_cover_mean',
 'dew_point_2m_mean',
 'wind_speed_10m_max',
 'wind_gusts_10m_max',
 'wind_direction_10m_dominant',
 'shortwave_radiation_sum',
 'et0_fao_evapotranspiration',
 'city',
 'month',
 'month_cos',
 'month_sin']

Let's add derived columns:

In [12]:
df['temp_range']=(df['temperature_2m_max']-df['temperature_2m_min'])
df['dew_point_depression']=(df['temperature_2m_mean']-df['dew_point_2m_mean'])
df['win_dir_sin']=np.sin(np.deg2rad(df['wind_direction_10m_dominant']))
df['win_dir_cos']=np.cos(np.deg2rad(df['wind_direction_10m_dominant']))
doy=df['date'].dt.day_of_year
df['doy_sin']=np.sin(2*np.pi*doy/365.25)
df['doy_cos']=np.cos(2*np.pi*doy/365.25)

Let's compute lags 

In [13]:
g=df.groupby('city')
for k in (1,3,7):
    df[f'rain_lag_{k}']=g['rain_sum'].shift(k)
df['pressure_change_1d']=g['surface_pressure_mean'].diff()
df['humidity_lag_1']=g['relative_humidity_2m_mean'].shift()

Let's compute rolling columns :

In [14]:
for w in (3,7,30):
    df[f'rain_roll_{w}_sum']=g['rain_sum'].transform(lambda s:s.shift(1).rolling(w).sum())

In [15]:
df=df.sort_values(['city','date'])
test=df.groupby('city',group_keys=False).apply(lambda g:g.tail(int (len(g)*0.2)))
df_train=df.drop(test.index)
df_test=df.loc[test.index]

In [52]:
cols_to_drop=['precipitation_sum', 'precipitation_hours', 'month','date','rain_sum']
df=df.dropna()
df['city']=df['city'].astype('category')


In [53]:
X_train=df_train.drop(cols_to_drop,axis=1)
X_test=df_test.drop(cols_to_drop,axis=1)
y_train=df_train['precipitation_sum']
y_test=df_test['precipitation_sum']
X_train['city']=X_train['city'].astype('category')
X_test['city']=X_test['city'].astype('category')

In [54]:
from xgboost import XGBRegressor
v0=XGBRegressor(n_estimators=600,learning_rate=0.03,max_depth=6,subsample=0.8,colsample_bytree=0.8,enable_categorical=True,
               tree_method='hist')

In [55]:
v0.fit(X_train,y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [56]:
y_pred_0=v0.predict(X_test)

In [57]:
from sklearn.metrics import mean_absolute_error 
mae_0=mean_absolute_error(y_pred_0,y_test)
(mae_0/y_test.mean())*100

np.float64(75.5694368467072)

Ok nice we got good model but way too many features let's see if we can drop some of them 

In [58]:
def get_feature_importance(model,features):
    df_importance=pd.DataFrame({'feature':features,
                              'importance':model.feature_importances_}).sort_values('importance',ascending=False).reset_index(drop=True)
    return df_importance

In [59]:
get_feature_importance(v0,X_train.columns.tolist())

,feature,importance
0,shortwave_radiation_sum,0.172137
1,dew_point_depression,0.091143
2,wind_gusts_10m_max,0.079672
3,rain_lag_1,0.074658
4,et0_fao_evapotranspiration,0.061618
5,surface_pressure_mean,0.045021
6,cloud_cover_mean,0.043531
7,city,0.039292
8,pressure_change_1d,0.033147
9,relative_humidity_2m_mean,0.032154
